In [4]:
import os
import sys
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score,
                             roc_auc_score)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from imblearn.over_sampling import SMOTE
from joblib import dump

# Optional: XGBoost
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False

# Config
DATA_DIR = "data"
OUTPUT_DIR = "outputs"
TRAIN_FILENAME = "/content/KDDTrain+.txt"
TEST_FILENAME = "/content/KDDTest+.txt"
RANDOM_STATE = 42
TEST_SIZE = 0.2
CATEGORICAL_COLS = [1, 2, 3]  # protocol, service, flag

os.makedirs(OUTPUT_DIR, exist_ok=True)


In [5]:
# ================================================================
# 2. LOAD DATA
# ================================================================
train_path = os.path.join(DATA_DIR, TRAIN_FILENAME)
test_path = os.path.join(DATA_DIR, TEST_FILENAME)

if os.path.exists(train_path) and os.path.exists(test_path):
    df_train = pd.read_csv(train_path, header=None)
    df_test = pd.read_csv(test_path, header=None)
    df = pd.concat([df_train, df_test], axis=0).reset_index(drop=True)
elif os.path.exists(test_path):
    df = pd.read_csv(test_path, header=None)
else:
    print("No NSL-KDD files found in data/.")
    sys.exit(1)

label_col = df.columns[-2]      # attack label
difficulty_col = df.columns[-1] # difficulty


In [6]:
# ================================================================
# 3. EDA
# ================================================================
print("=== EDA ===")
print("Shape:", df.shape)
print("\nHead:\n", df.head())
print("\nInfo:")
print(df.info())
print("\nDescribe:\n", df.describe().T)

attack_counts = df[label_col].value_counts()
plt.figure(figsize=(10, 6))
attack_counts.plot(kind="bar")
plt.title("Attack Type Distribution")
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "attack_distribution.png"))
plt.close()
print("Saved attack_distribution.png")

# ================================================================
# 4. PREPROCESSING
# ================================================================
y = df[label_col].astype(str).copy()
X = df.drop(columns=[label_col, difficulty_col])

# Remove duplicates
X = X.drop_duplicates()
y = y.loc[X.index]

# Missing values
if X.isnull().sum().sum() > 0:
    X = X.fillna(0)

# Split categorical & numeric
cat_cols = [c for c in CATEGORICAL_COLS if c in X.columns]
num_cols = [c for c in X.columns if c not in cat_cols]

# Encode categorical
X_cat = pd.get_dummies(X[cat_cols].astype(str), prefix=["p", "s", "f"])
X_num = X[num_cols].apply(pd.to_numeric, errors="coerce").fillna(0)

# Merge and fix column names
X_processed = pd.concat(
    [X_num.reset_index(drop=True), X_cat.reset_index(drop=True)], axis=1
)
X_processed.columns = X_processed.columns.map(str)

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)
pd.Series(dict(enumerate(le.classes_))).to_csv(
    os.path.join(OUTPUT_DIR, "label_map.csv"), header=False
)

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_processed)
dump(scaler, os.path.join(OUTPUT_DIR, "scaler.joblib"))

X = pd.DataFrame(X_scaled, columns=X_processed.columns)
y = y_encoded


=== EDA ===
Shape: (148517, 43)

Head:
    0    1         2   3    4     5   6   7   8   9   ...    33    34    35  \
0   0  tcp  ftp_data  SF  491     0   0   0   0   0  ...  0.17  0.03  0.17   
1   0  udp     other  SF  146     0   0   0   0   0  ...  0.00  0.60  0.88   
2   0  tcp   private  S0    0     0   0   0   0   0  ...  0.10  0.05  0.00   
3   0  tcp      http  SF  232  8153   0   0   0   0  ...  1.00  0.00  0.03   
4   0  tcp      http  SF  199   420   0   0   0   0  ...  1.00  0.00  0.00   

     36    37    38    39    40       41  42  
0  0.00  0.00  0.00  0.05  0.00   normal  20  
1  0.00  0.00  0.00  0.00  0.00   normal  15  
2  0.00  1.00  1.00  0.00  0.00  neptune  19  
3  0.04  0.03  0.01  0.00  0.01   normal  21  
4  0.00  0.00  0.00  0.00  0.00   normal  21  

[5 rows x 43 columns]

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 148517 entries, 0 to 148516
Data columns (total 43 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   

In [10]:
# ================================================================
# PREPROCESSING
# ================================================================

# y = labels (attack type)
y = df[df.columns[-2]].astype(str).copy()   # second last column = attack label

# X = features (drop label + difficulty)
X = df.drop(columns=[df.columns[-2], df.columns[-1]])

# --- Remove duplicates ---
X = X.drop_duplicates()
y = y.loc[X.index]

# --- Handle missing values ---
if X.isnull().sum().sum() > 0:
    X = X.fillna(0)

# --- Split categorical & numeric ---
cat_cols = [1, 2, 3]  # protocol_type, service, flag (indices)
num_cols = [c for c in X.columns if c not in cat_cols]

# --- Encode categorical features ---
X_cat = pd.get_dummies(X[cat_cols].astype(str), prefix=["protocol", "service", "flag"])
X_num = X[num_cols].apply(pd.to_numeric, errors="coerce").fillna(0)

# --- Combine ---
X_processed = pd.concat(
    [X_num.reset_index(drop=True), X_cat.reset_index(drop=True)], axis=1
)

# ✅ Make all column names strings (important for scaler)
X_processed.columns = X_processed.columns.map(str)

print("Processed features shape:", X_processed.shape)

# --- Encode labels ---
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Classes found:", len(le.classes_))
pd.Series(dict(enumerate(le.classes_))).to_csv(
    os.path.join(OUTPUT_DIR, "label_map.csv"), header=False
)

# --- Scale numerical features ---
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_processed)

import joblib
joblib.dump(scaler, os.path.join(OUTPUT_DIR, "scaler.joblib"))

# Final DataFrames
X = pd.DataFrame(X_scaled, columns=X_processed.columns)
y = y_encoded

print("Final shapes -> X:", X.shape, " y:", y.shape)


Processed features shape: (147790, 122)
Classes found: 40
Final shapes -> X: (147790, 122)  y: (147790,)


In [14]:
from imblearn.over_sampling import SMOTE

print("Before balancing:", np.bincount(y_train))

try:
    sm = SMOTE(random_state=42, k_neighbors=2)
    X_train, y_train = sm.fit_resample(X_train, y_train)
    print("After SMOTE:", np.bincount(y_train))
except ValueError:
    print("⚠️ Too few samples in some classes, switching to RandomOverSampler.")
    from imblearn.over_sampling import RandomOverSampler
    ros = RandomOverSampler(random_state=42)
    X_train, y_train = ros.fit_resample(X_train, y_train)
    print("After RandomOverSampler:", np.bincount(y_train))



Before balancing: [  590  1040    40     9  1027   106    10  2909    15     9   234   797
    20    14 36573  1251 61537     4     5   177  2455   548    12    18
   250  3485    11  2486   104   265     2     2   721     2   712   770
     2     7     3    10]
⚠️ Too few samples in some classes, switching to RandomOverSampler.
After RandomOverSampler: [61537 61537 61537 61537 61537 61537 61537 61537 61537 61537 61537 61537
 61537 61537 61537 61537 61537 61537 61537 61537 61537 61537 61537 61537
 61537 61537 61537 61537 61537 61537 61537 61537 61537 61537 61537 61537
 61537 61537 61537 61537]


In [21]:
# ================================================================
# FAST PIPELINE (Quick Experiments) - WITH TIMER & MODE SWITCH
# ================================================================

import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score
from imblearn.over_sampling import RandomOverSampler
from sklearn.preprocessing import LabelEncoder, StandardScaler

# --- Step 1: Use a sample of dataset ---
df_sample = df.sample(n=20000, random_state=42)  # adjust n for speed

# --- Step 2: Preprocess ---
y = df_sample[df_sample.columns[-2]].astype(str)   # attack label
X = df_sample.drop(columns=[df_sample.columns[-2], df_sample.columns[-1]])  # drop label + difficulty

# Convert categorical indices -> column names
cat_cols = [1, 2, 3]  # protocol, service, flag
cat_colnames = [X.columns[i] for i in cat_cols]

# One-hot encode categoricals
X_cat = pd.get_dummies(X[cat_colnames].astype(str), prefix=["protocol", "service", "flag"])

# Numeric features
X_num = X.drop(columns=cat_colnames).apply(pd.to_numeric, errors="coerce").fillna(0)

# Merge
X_processed = pd.concat([X_num.reset_index(drop=True), X_cat.reset_index(drop=True)], axis=1)
X_processed.columns = X_processed.columns.map(str)

# Encode labels
le = LabelEncoder()
y = le.fit_transform(y)

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_processed)

# --- Step 2.5: Drop classes with < 2 samples (avoid stratify error) ---
class_counts = np.bincount(y)
valid_classes = np.where(class_counts > 1)[0]
mask = np.isin(y, valid_classes)

X_scaled = X_scaled[mask]
y = y[mask]

# --- Step 3: Train-Test Split ---
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=42
)

# --- Step 4: Balance with RandomOverSampler ---
ros = RandomOverSampler(random_state=42)
X_train, y_train = ros.fit_resample(X_train, y_train)

# --- Step 5: Define Models ---
MODE = "fast"   # change to "full" for RandomForest too

models = {
    "LogisticRegression": LogisticRegression(max_iter=500, random_state=42),
    "DecisionTree": DecisionTreeClassifier(max_depth=15, random_state=42),
}

if MODE == "full":
    models["RandomForest"] = RandomForestClassifier(
        n_estimators=50, max_depth=20, n_jobs=-1, random_state=42
    )

# --- Step 6: Train & Evaluate ---
results = []
for name, model in models.items():
    print(f"\n🔹 Training {name} ...")
    start = time.time()

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    elapsed = time.time() - start
    f1 = f1_score(y_test, preds, average="macro")

    print(f"{name} F1-score (macro): {f1:.4f}")
    print(f"⏱️ Training + prediction time: {elapsed:.2f} sec")
    print(classification_report(y_test, preds, zero_division=0))

    results.append({"model": name, "f1_macro": f1, "time_sec": elapsed})

print("\n=== Fast Experiment Results ===")
print(pd.DataFrame(results).sort_values(by="f1_macro", ascending=False))



🔹 Training LogisticRegression ...
LogisticRegression F1-score (macro): 0.7275
⏱️ Training + prediction time: 321.47 sec
              precision    recall  f1-score   support

           0       1.00      0.95      0.97        20
           1       0.87      0.97      0.92        35
           2       0.00      0.00      0.00         1
           3       0.94      1.00      0.97        32
           4       1.00      1.00      1.00         3
           5       0.00      0.00      0.00         0
           6       0.97      0.96      0.96        96
           7       1.00      1.00      1.00         2
           9       0.47      1.00      0.64         7
          10       0.93      1.00      0.97        28
          12       0.00      0.00      0.00         1
          13       1.00      1.00      1.00      1230
          14       0.85      0.98      0.91        45
          15       1.00      0.92      0.96      2082
          17       1.00      1.00      1.00         7
          18  

In [23]:
from sklearn.metrics import accuracy_score

# --- Step 6: Train & Evaluate ---
results = []
for name, model in models.items():
    print(f"\n🔹 Training {name} ...")
    start = time.time()

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    elapsed = time.time() - start
    f1 = f1_score(y_test, preds, average="macro")
    acc = accuracy_score(y_test, preds)

    print(f"{name} Accuracy: {acc:.4f}")
    print(f"{name} F1-score (macro): {f1:.4f}")
    print(f"⏱️ Training + prediction time: {elapsed:.2f} sec")
    print(classification_report(y_test, preds, zero_division=0))

    results.append({
        "model": name,
        "accuracy": acc,
        "f1_macro": f1,
        "time_sec": elapsed
    })

print("\n=== Fast Experiment Results ===")
print(pd.DataFrame(results).sort_values(by="f1_macro", ascending=False))



🔹 Training LogisticRegression ...
LogisticRegression Accuracy: 0.9487
LogisticRegression F1-score (macro): 0.7275
⏱️ Training + prediction time: 322.71 sec
              precision    recall  f1-score   support

           0       1.00      0.95      0.97        20
           1       0.87      0.97      0.92        35
           2       0.00      0.00      0.00         1
           3       0.94      1.00      0.97        32
           4       1.00      1.00      1.00         3
           5       0.00      0.00      0.00         0
           6       0.97      0.96      0.96        96
           7       1.00      1.00      1.00         2
           9       0.47      1.00      0.64         7
          10       0.93      1.00      0.97        28
          12       0.00      0.00      0.00         1
          13       1.00      1.00      1.00      1230
          14       0.85      0.98      0.91        45
          15       1.00      0.92      0.96      2082
          17       1.00      1.0

In [22]:
# --- Step 7: Save the Best Model ---
import joblib

# Pick the best model based on F1-macro
best_result = max(results, key=lambda x: x["f1_macro"])
best_model_name = best_result["model"]
best_model = models[best_model_name]

print(f"\n🏆 Best Model: {best_model_name} with F1-macro={best_result['f1_macro']:.4f}")

# Save model and preprocessing steps
joblib.dump(best_model, f"{best_model_name}_model.joblib")
joblib.dump(le, "label_encoder.joblib")
joblib.dump(scaler, "scaler.joblib")

print("✅ Model and preprocessing objects saved!")




🏆 Best Model: LogisticRegression with F1-macro=0.7275
✅ Model and preprocessing objects saved!
